## WaveNet-style CNN

Below is an implementation of a model following the "wave" architecture as seen in Google DeepMind's [WaveNet](https://arxiv.org/pdf/1609.03499) paper. We start with a sequence of 8 tokens, and slowly flatten it to 

In [1]:
from datasets import load_dataset

ds = load_dataset("parquet", 
                    data_files={'train': 'data/train.parquet', 
                                "validation" : "data/validation.parquet", 
                                'test': 'data/test.parquet'}
                    )

c:\Users\n_mac\Desktop\Coding Portfolio\sentencenet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text_split = []

for row in ds["train"]["sentence"]:
    for w in row.split():
        text_split.append(w)
        
text_split = set(text_split)

In [3]:
s_i = {s:i+1 for i, s in enumerate(text_split)}
s_i["<n>"] = 0

i_s = {i:s for s, i in s_i.items()}

In [4]:
import torch

# building dataset

block_size = 8

def build_dataset(sents):
    
    X, Y = [], []
    for s in sents:
        #print(s)
        context = [0] * block_size
        s[-1] = "<n>"
        
        for w in s:
            
            ix = s_i[w]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]
            
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [5]:
sents_split = [s.split() for s in ds["train"]["sentence"]]

Xtr, ytr = build_dataset(sents_split)


In [9]:
Xtr.shape

torch.Size([887521, 8])

In [7]:
dstr = torch.utils.data.TensorDataset(Xtr, ytr)

trloader = torch.utils.data.DataLoader(
    dataset=dstr,
    batch_size=500,
    shuffle=True,
    
)

In [8]:
class Reshape(torch.nn.Module):    
    def __call__(self, x):
        # print(x.shape[0] / 2)
        if x.dim() == 2:
            return x.view((int(x.shape[0] /2), -1))
        else:
            return torch.reshape(x, (int(x.shape[0]), int(x.shape[1] / 2), -1))
        
class Squeeze(torch.nn.Module):
    def __call__(self, x):
        return torch.squeeze(x)

class NN(torch.nn.Module):
    def __init__(self, vocab_size, emb_dim, n_hidden):
        super().__init__()
        
        self.vocab_size = vocab_size
        
        self.layers = [
            torch.nn.Embedding(vocab_size, emb_dim),
            torch.nn.Linear(emb_dim, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Squeeze(), torch.nn.Linear(n_hidden, vocab_size)
        ]
        
        self.out = 0.0
   
        params = []
        
        for layer in self.layers:
            for p in layer.parameters():
                params.append(p)
                p.retain_grad()
        
        self.parameters_ = params
        
    """ def parameters(self):
        params = []
        
        for layer in self.layers:
            for p in layer.parameters:
                params.append(p)
        
        return params """
    
    def __call__(self, x):
        _x = x
        
        for layer in self.layers:
            # print(type(layer))
            # print(_x.shape)
            _x = layer(_x)
            
            
        
        self.out = _x
        return _x
    
    def fit(self, max_iter, loader, lr):
        g = torch.Generator().manual_seed(2147483647)
        optimizer = torch.optim.AdamW(self.parameters_, lr=lr)
        
        lossi = []
        
        for p in self.parameters_:
            p.retain_grad()
        
        for step in range(max_iter):
            Xb, yb = next(iter(loader))
            logits = self.__call__(Xb)
            
            loss = torch.nn.functional.cross_entropy(logits, yb)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            lossi.append(loss.item())
            
            for p in self.parameters_:
                p = p - lr * p.grad
                
            if step % (max_iter / 10) == 0:
                print(f"{step} / {max_iter}: {loss:.6f}")
        
        return lossi
                

In [62]:
embedding_dim = 32

n_hidden = 100
vocab_size = len(s_i)

net = NN(vocab_size=vocab_size, emb_dim=embedding_dim, n_hidden=n_hidden)


In [63]:
net.fit(max_iter=10000,  loader=trloader, lr=1e-3);

0 / 10000: 9.258332
1000 / 10000: 5.504152
2000 / 10000: 5.426430
3000 / 10000: 5.192110
4000 / 10000: 5.254045
5000 / 10000: 5.031169
6000 / 10000: 5.086753
7000 / 10000: 4.782856
8000 / 10000: 5.049802
9000 / 10000: 4.940383


In [64]:
Xval, yval = build_dataset(s.split() for s in ds["validation"]["sentence"])
dsval = torch.utils.data.TensorDataset(Xval, yval)

valloader = torch.utils.data.DataLoader(dsval, batch_size=100)

In [65]:
def eval_model(model, loader):
    total_loss = 0.0
    
    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            loss = torch.nn.functional.cross_entropy(logits, y)
            
            total_loss += loss.item() * x.size(0)
            
    return total_loss / len(loader.dataset)

In [67]:
print("Training set loss:", eval_model(net, trloader))
print("Validation set loss:", eval_model(net, valloader))

Training set loss: 4.836368589899691
Validation set loss: 5.253106556217818


In [66]:
g2 = torch.Generator().manual_seed(2147483647 + 1)

for _ in range(5):
    out = []
    context = [0] * block_size
    
    while True:
        probs = net(torch.tensor(context))
        logits = torch.nn.functional.softmax(probs, dim=0)

        ix = torch.multinomial(logits, num_samples=1, generator=g2).item()

        context = context[1:] + [ix]
        
        if ix == 0:
                    break
        
        out.append(ix)
        # print(i_s[ix])
        

    print("".join(i_s[i] + " " for i in out))
    

but then no sign we are no step allowed the second flow of my sources 
there 's recently are opposed mr. <unk> said <unk> does n't generates <unk> seven 
the <unk> familiar with daiwa laboratory is tested as regarded as lead of plaintiffs from american times while inadequate stock and 
the tokyo report fell N points in an elderly 
the charges includes N N east partnership has lost N N to N N N senior subordinated one-year buy-out preferred transport debt via <unk> at prudential-bache 
